# Hawkes-GRU Native Adapter v1

目标：把 `hawkes_gru_mse` 从 event-decay Ridge proxy 推进到 `src/model` 的 native PyTorch sequence adapter。Notebook 只保留实验流程和中文结论；可复用训练、回测、analysis 逻辑都调用 `src/`。外部论文只保存 URL 与自有摘要，不复制全文或第三方代码。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.sequence import HawkesGRUSequenceRegressor
from src.model.training import fit_estimator_for_model, model_adapter_name

OUTPUT_DIR = REPO_ROOT / 'research' / 'hawkes_gru_native_adapter_v1'
CARD_DIR = OUTPUT_DIR / 'matrix_card'

outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'gru_mse', 'hawkes_gru_mse', 'sfm_mse', 'tcn_mse'),
)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
feature_columns = tuple(dataset.feature_columns[:5])
estimator, constant_prediction = fit_estimator_for_model('hawkes_gru_mse', train=train, feature_columns=feature_columns)
capability = model_capability_table(('hawkes_gru_mse',)).iloc[0].to_dict()
fidelity = model_fidelity_summary()

assert constant_prediction is None
assert isinstance(estimator, HawkesGRUSequenceRegressor)
assert estimator.lookback_ == 5
assert estimator.feature_count_ == 5
assert estimator.hidden_size == 8
assert hasattr(estimator, 'model_')
assert hasattr(estimator.model_, 'intensity_decay')
assert hasattr(estimator.model_, 'event_gate')
assert capability['paper_fidelity'] == 'native_torch'
assert capability['is_proxy'] is False
assert model_adapter_name('hawkes_gru_mse') == 'hawkes_gru_intensity_mse_v1'
assert 'hawkes_gru_mse' in fidelity['native_models']
assert 'hawkes_gru_mse' not in fidelity['proxy_models']

summary = {
    'schema_version': 'research.hawkes_gru_native_adapter_v1',
    'status': 'completed',
    'model': 'hawkes_gru_mse',
    'adapter': model_adapter_name('hawkes_gru_mse'),
    'estimator': estimator.__class__.__name__,
    'lookback': estimator.lookback_,
    'feature_count': estimator.feature_count_,
    'hidden_size': estimator.hidden_size,
    'paper_fidelity': capability['paper_fidelity'],
    'is_proxy': bool(capability['is_proxy']),
    'full_matrix_native_model_count': len(fidelity['native_models']),
    'full_matrix_proxy_model_count': len(fidelity['proxy_models']),
    'matrix_card': str(CARD_DIR.relative_to(REPO_ROOT)),
    'paper_reference': {
        'source_id': 'arxiv:1612.09328',
        'url': 'https://arxiv.org/abs/1612.09328',
        'usage': '自有摘要和结构假设；实现只复用 Hawkes-style event intensity 与 recurrent sequence 思路，不复制论文全文或第三方代码。',
    },
    'artifact_links': {
        'analysis_summary': str((CARD_DIR / 'analysis_summary.json').relative_to(REPO_ROOT)),
        'model_comparison': str((CARD_DIR / 'model_comparison.csv').relative_to(REPO_ROOT)),
        'model_run_ledger': str((CARD_DIR / 'model_run_ledger.csv').relative_to(REPO_ROOT)),
        'manifest': str((CARD_DIR / 'analysis_manifest.json').relative_to(REPO_ROOT)),
    },
}
(OUTPUT_DIR / 'notebook_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.hawkes_gru_native_adapter_v1',
 'status': 'completed',
 'model': 'hawkes_gru_mse',
 'adapter': 'hawkes_gru_intensity_mse_v1',
 'estimator': 'HawkesGRUSequenceRegressor',
 'lookback': 5,
 'feature_count': 5,
 'hidden_size': 8,
 'paper_fidelity': 'native_torch',
 'is_proxy': False,
 'full_matrix_native_model_count': 20,
 'full_matrix_proxy_model_count': 12,
 'matrix_card': 'research/hawkes_gru_native_adapter_v1/matrix_card',
 'paper_reference': {'source_id': 'arxiv:1612.09328',
  'url': 'https://arxiv.org/abs/1612.09328',
  'usage': '自有摘要和结构假设；实现只复用 Hawkes-style event intensity 与 recurrent sequence 思路，不复制论文全文或第三方代码。'},
 'artifact_links': {'analysis_summary': 'research/hawkes_gru_native_adapter_v1/matrix_card/analysis_summary.json',
  'model_comparison': 'research/hawkes_gru_native_adapter_v1/matrix_card/model_comparison.csv',
  'model_run_ledger': 'research/hawkes_gru_native_adapter_v1/matrix_card/model_run_ledger.csv',
  'manifest': 'research/hawkes_gru_nati

## 中文结论

- 做了什么：在 `src/model.sequence` 增加 `HawkesGRUSequenceRegressor`，把 `hawkes_gru_mse` 从 `proxy_feature_adapter` 升级为 `native_torch`。实现保留 Hawkes-style event intensity 的核心结构假设：用相邻时间步的特征冲击形成事件强度，按可学习指数衰减累计 intensity，再用 event gate 调制 GRU 输入。
- 核心指标：本 notebook 训练并回测 `ridge_mse`、`gru_mse`、`hawkes_gru_mse`、`sfm_mse`、`tcn_mse`，输出 `model_comparison.csv`、`daily_backtest.csv`、`model_run_ledger.csv`、`qlib_risk_metrics.csv`、`model_effect_deltas.csv`、`model_regime_window_diagnostics.csv` 和 manifest。具体收益、回撤、Sharpe 以 `notebook_summary.json` 与 `matrix_card/analysis_summary.json` 为准。
- regime 差异：本轮复用 `src/analysis` 的 window/regime、qlib risk、model effect delta 和模型级 regime/window 诊断；Hawkes-GRU 是否有价值要重点看 stress/downtrend 下事件冲击后的相对收益、回撤、稳定性和换手。
- 是否晋升：否。`native_torch` 只表示实现形态已从 proxy 推进为稳定训练接口；production 晋升仍需要更长历史窗口、更多市场 regime、成本压力和 paper/sim replay 复验。
- 风险：当前是紧凑实现，不等价于完整 Neural Hawkes 训练规模；SOXL 单标的样本短，事件强度门控可能过拟合短期冲击；外部论文仅作为结构参考，未复制第三方代码或全文。
- 下一步：刷新全模型 ledger，使 native/proxy 计数同步；继续替换 graph proxy 或 `sequence_lag_ridge_mse`/`ridge_ic` 等剩余 proxy。